In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Healthcare Data Surge Optimization") \
    .getOrCreate()

In [2]:
df = spark.read.csv("healthcare_data.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

+----------+-------------------+----------+------------+-----------+------------+
|patient_id|          timestamp|heart_rate|oxygen_level|systolic_bp|diastolic_bp|
+----------+-------------------+----------+------------+-----------+------------+
|      1051|2026-01-01 00:00:00|        93|          91|        104|          75|
|      1092|2026-01-01 00:00:01|       106|          94|        128|          70|
|      1014|2026-01-01 00:00:02|        67|          95|        136|          72|
|      1071|2026-01-01 00:00:03|        99|          93|        119|          77|
|      1060|2026-01-01 00:00:04|       108|          93|        120|          70|
+----------+-------------------+----------+------------+-----------+------------+
only showing top 5 rows
root
 |-- patient_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- heart_rate: integer (nullable = true)
 |-- oxygen_level: integer (nullable = true)
 |-- systolic_bp: integer (nullable = true)
 |-- diastolic

In [3]:
from pyspark.sql.functions import col, expr

# Duplicate dataset
df_double = df.union(df)

# Adjust timestamps slightly (simulate higher frequency)
df_double = df_double.withColumn(
    "timestamp",
    expr("timestamp + interval 1 seconds")
)

df_double.show(5)

+----------+-------------------+----------+------------+-----------+------------+
|patient_id|          timestamp|heart_rate|oxygen_level|systolic_bp|diastolic_bp|
+----------+-------------------+----------+------------+-----------+------------+
|      1051|2026-01-01 00:00:01|        93|          91|        104|          75|
|      1092|2026-01-01 00:00:02|       106|          94|        128|          70|
|      1014|2026-01-01 00:00:03|        67|          95|        136|          72|
|      1071|2026-01-01 00:00:04|        99|          93|        119|          77|
|      1060|2026-01-01 00:00:05|       108|          93|        120|          70|
+----------+-------------------+----------+------------+-----------+------------+
only showing top 5 rows


In [4]:
from pyspark.sql.functions import when

df_processed = df_double.withColumn(
    "critical_flag",
    when(
        (col("heart_rate") > 110) |
        (col("oxygen_level") < 92) |
        (col("systolic_bp") > 135),
        1
    ).otherwise(0)
)

df_processed.show(10)

+----------+-------------------+----------+------------+-----------+------------+-------------+
|patient_id|          timestamp|heart_rate|oxygen_level|systolic_bp|diastolic_bp|critical_flag|
+----------+-------------------+----------+------------+-----------+------------+-------------+
|      1051|2026-01-01 00:00:01|        93|          91|        104|          75|            1|
|      1092|2026-01-01 00:00:02|       106|          94|        128|          70|            0|
|      1014|2026-01-01 00:00:03|        67|          95|        136|          72|            1|
|      1071|2026-01-01 00:00:04|        99|          93|        119|          77|            0|
|      1060|2026-01-01 00:00:05|       108|          93|        120|          70|            0|
|      1020|2026-01-01 00:00:06|       103|          93|        109|          87|            0|
|      1082|2026-01-01 00:00:07|        78|          91|        104|          71|            1|
|      1086|2026-01-01 00:00:08|       1

In [5]:
from pyspark.sql.functions import avg, count

dashboard_df = df_processed.groupBy("patient_id").agg(
    avg("heart_rate").alias("avg_heart_rate"),
    avg("oxygen_level").alias("avg_oxygen"),
    count("*").alias("total_records"),
)

dashboard_df.show()

+----------+-----------------+-----------------+-------------+
|patient_id|   avg_heart_rate|       avg_oxygen|total_records|
+----------+-----------------+-----------------+-------------+
|      1088|             91.0|             95.0|           28|
|      1084|           96.625|             93.5|           16|
|      1025|91.77777777777777|94.22222222222223|           18|
|      1005|            94.25|           93.875|           16|
|      1016|             92.2|93.06666666666666|           30|
|      1068|75.58333333333333|             94.5|           24|
|      1031|93.83333333333333|94.33333333333333|           24|
|      1051|92.16666666666667|93.33333333333333|           24|
|      1034|             93.3|             93.6|           20|
|      1064|            80.25|             96.0|           16|
|      1030|82.33333333333333|             94.0|            6|
|      1019|93.57142857142857|93.28571428571429|           14|
|      1085|90.73333333333333|95.33333333333333|       